# Qian: BitFit cosine-head resolution ablation

Compare 336 vs 384 vs 448 on the same 1,000 classes, 40 fit + 5 holdout images/class, seed 42. Train all three fresh for three epochs with identical batch size and optimizer settings. Build separate fitting-only prototypes at each resolution. Select best.pt using crop 0.9 without flip; evaluate each checkpoint at crop 0.75 with and without flip. Record accuracy, memory and training throughput. Equal epochs compare data exposure; follow up with equal time if a higher resolution wins. GPU required; run all.


In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "qq_bitfit_ablation"
TEAM = "08"
CONFIG = "configs/bitfit_resolution_ablation.yaml"
EXPERIMENT_NAME = "qq_bitfit_resolution_336_384_448_seed42_v1"
EXPERIMENT_DIR = f"{DRIVE_OUT}/{EXPERIMENT_NAME}"
SPLIT_DIR = f"/content/{EXPERIMENT_NAME}_splits"
# Use a new name if changing the recipe or epoch count.
RESOLUTIONS = (336, 384, 448)


## 1. Clone or pull the repo

In [ ]:
import base64
import os
import subprocess
from getpass import getpass

# Pass credentials through a temporary child-process environment, not the URL.
token = getpass("GitHub PAT (hidden; repo scope). Leave empty for a public repo: ")
git_env = os.environ.copy()
if token:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {auth}")
    del auth
try:
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], env=git_env, check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], env=git_env, check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], env=git_env, check=True)
finally:
    del token, git_env
os.chdir(REPO_DIR)
actual = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
assert actual == BRANCH, f"Wrong branch: {actual}"
print("Branch:", actual)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
subprocess.run(["python", "scripts/download_data.py", "--dest", DATA_DIR], check=True)


## 5. Train cosine-head BitFit at each resolution

Three epochs each, batch 64, bias/head LR 1e-3, zero weight decay, BF16, 5% warmup + cosine. All three arms use the same split and seed. Prototypes use fitting images only and are stored by resolution. If memory requires a smaller batch, change batch and accumulation for all three arms and use a new experiment name.


In [ ]:
import torch
from src.utils import load_config, apply_overrides
from src.peft.bitfit_ablation import prepare, stage_config, ensure_prototypes, run_arm, write_results
import pandas as pd

assert torch.cuda.is_available(), "Select a CUDA GPU runtime."
BASE = load_config(CONFIG)
apply_overrides(BASE, data_root=DATA_DIR, output_dir=EXPERIMENT_DIR)
EXPERIMENT = Path(EXPERIMENT_DIR)
EXPERIMENT.mkdir(parents=True, exist_ok=True)
ROOTS = prepare(BASE, EXPERIMENT, split_dir=SPLIT_DIR)
results = []
for resolution in RESOLUTIONS:
    arm = {"arm": f"all_biases_cosine_{resolution}", "method": "bitfit", "bitfit_scope": "all", "head_type": "cosine", "img_size": resolution}
    cfg = stage_config(BASE, EXPERIMENT, ROOTS["pilot"], "resolution", arm, 42, 1e-3)
    ensure_prototypes(cfg, allow_missing_classes=True)
    result = run_arm(cfg)
    results.append({**result, "head_type": "cosine", "img_size": resolution})
write_results(EXPERIMENT, "resolution", results)
display(pd.DataFrame(results))
for result in results:
    subprocess.run(["python", "scripts/plot_curves.py", "--run_dir", result["run_dir"],
                    "--out_dir", str(Path(result["run_dir"]) / "figures")], check=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
(EXPERIMENT / "launcher_commit.txt").write_text(commit + "\n")


## 6. Holdout evaluation: crop 0.75 with/without flip

Also evaluate crop 0.9 without flip to reproduce checkpoint-selection accuracy. Results are development scores; one seed and three epochs do not establish convergence. No CodaBench submissions.


In [ ]:
import gc
import math
from torch.utils.data import DataLoader
from src.data import ImageListDataset, build_transforms
from src.model import build_model
from src.predict import load_checkpoint
from src.utils import get_device, set_seed

@torch.no_grad()
def score_holdout(model, cfg, crop, flip, device):
    _, transform = build_transforms(cfg, eval_crop_pct=crop)
    dataset = ImageListDataset(cfg.data_root, "val", transform=transform)
    if not dataset.has_labels:
        raise ValueError("This sweep requires a labeled internal holdout.")
    loader = DataLoader(dataset, batch_size=int(cfg.batch_size), shuffle=False,
                        num_workers=int(cfg.num_workers), pin_memory=device.type == "cuda")
    correct, count, nll = 0, 0, 0.0
    for images, labels, _ in loader:
        images, labels = images.to(device), labels.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16,
                            enabled=bool(cfg.bf16) and device.type == "cuda"):
            logits = model(images)
            mirrored = model(torch.flip(images, dims=[3])) if flip else None
        log_probs = logits.float().log_softmax(dim=1)
        if flip:
            log_probs = torch.logaddexp(log_probs, mirrored.float().log_softmax(dim=1)) - math.log(2)
        correct += int((log_probs.argmax(1) == labels).sum())
        count += labels.numel()
        nll += float(torch.nn.functional.nll_loss(log_probs, labels, reduction="sum"))
    return {"val_acc": correct / count, "val_nll": nll / count, "holdout_images": count}

rows = []
device = get_device()
for result in results:
    run_dir = Path(result["run_dir"])
    checkpoint = run_dir / "best.pt"
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
    cfg = load_config(run_dir / "config.yaml")
    # Reattach current-session image symlinks; persisted manifests use the same split.
    cfg.data_root = str(ROOTS["pilot"])
    set_seed(int(cfg.seed))
    model = build_model(cfg).to(device)
    load_checkpoint(model, checkpoint, device)
    model.eval()
    for crop in (0.9, 0.75):
        for flip in ((False,) if crop == 0.9 else (False, True)):
            metrics = score_holdout(model, cfg, crop, flip, device)
            rows.append({"img_size": result["img_size"], "head_type": result["head_type"], "crop_pct": crop, "tta_flip": flip,
                         "best_epoch": result["best_epoch"], "checkpoint": str(checkpoint), **metrics})
            print(rows[-1])
            if crop == 0.9 and not flip and abs(metrics["val_acc"] - result["best_val_acc"]) > 0.001:
                raise RuntimeError("Baseline differs from recorded best accuracy; check the split/checkpoint.")
            pd.DataFrame(rows).to_csv(EXPERIMENT / "resolution_holdout_results.csv", index=False)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
table = pd.DataFrame(rows)
display(table.sort_values("val_acc", ascending=False))
display(table.pivot(index="crop_pct", columns=["img_size", "tta_flip"], values="val_acc"))
